# 조달업체 CSV 전처리 — 벤더 RAG DB 적재용

목표: 나라장터 조달업체 CSV를 **RAG DB(pgvector)에 넣을 수 있는 형태**로 정리.

- 채울 수 있는 컬럼(회사명·사업자번호·주소·업종·취급품목)은 다 살림
- 못 채우는 컬럼(이메일·연락처·공급업체유형)은 **일부러 빈 값으로 남김** → 다음 단계(Tavily+추출모델 DB클렌징)에서 채울 대상이라는 표시
- `description`은 **품목명으로 검색했을 때 매칭이 잘 되도록**, 취급품목을 중심으로 구성함


In [ ]:
import pandas as pd
import io
import re

pd.set_option("display.max_columns", None)

## 1. 파일 불러오기

원본이 UTF-16(LE)로 저장돼있어서, bytes로 먼저 읽고 디코딩함.

In [ ]:
with open("조달업체_등록내역.csv", "rb") as f:
    raw = f.read()

text = raw.decode("utf-16-le", errors="replace")
df = pd.read_csv(io.StringIO(text))

print(f"원본: {len(df)}행, {len(df.columns)}컬럼")
df.head()

## 2. 필요한 컬럼만 골라서 이름 정리

In [ ]:
df = df[[
    "업체명", "사업자등록번호", "업체소재시군구", "업체국가",
    "기업구분", "대표업종", "대표세부품명", "나라장터등록일자",
]].rename(columns={
    "업체명": "company_name",
    "사업자등록번호": "business_number",
    "업체소재시군구": "address",
    "업체국가": "country",
    "기업구분": "company_size",       # 중소/대기업 (법적형태 아님, 참고용으로만 보관)
    "대표업종": "industry",
    "대표세부품명": "item_category",
    "나라장터등록일자": "registered_date",
})
df.head()

## 3. 국내/해외 구분

정식 사업자등록번호(10자리 숫자)가 아니라 `F`로 시작하면 해외기업 대체식별자.
국세청 사업자번호 진위확인 API는 국내기업에만 쓸 수 있어서 구분해둠.


In [ ]:
df["is_domestic"] = df["business_number"].astype(str).str.match(r"^\d{10}$")

before = len(df)
df = df[df["is_domestic"]].copy()   # 국내기업만. 해외기업도 필요하면 이 줄 빼면 됨
print(f"국내기업만 필터링: {before}행 → {len(df)}행")

## 4. 회사 기준 중복 제거 (⚡ 빠른 버전)

같은 회사가 취급품목마다 한 줄씩 등록되어 있어서(예: '힛더마크'가 2줄),
회사 단위로 묶어야 함. `lambda`를 쓰면 65만 행에서 몇 분씩 걸리므로,
**pandas 내장 집계함수(`.first()`, `.unique()`, `.min()`)만 써서** 속도를 확보함.

핵심: `business_number` 하나만 고유키로 쓰고(5개 컬럼 묶어서 그룹핑할 필요 없음),
결측치는 그룹핑 *전에* 미리 제거해서 lambda 없이 처리.


In [ ]:
# 회사당 대표값(첫 값)만 필요한 컬럼들 — .first()는 내장 최적화 함수라 빠름
company_info = df.groupby("business_number")[
    ["company_name", "address", "country", "company_size"]
].first()

# 여러 개 모아야 하는 컬럼들 — 결측치를 미리 빼고 .unique()로 (이것도 내장 최적화 함수)
item_categories = (
    df.dropna(subset=["item_category"])
    .groupby("business_number")["item_category"]
    .unique()
    .rename("item_categories")
)
industries = (
    df.dropna(subset=["industry"])
    .groupby("business_number")["industry"]
    .unique()
    .rename("industries")
)
registered_date = df.groupby("business_number")["registered_date"].min()

# 합치기
grouped = company_info.join([item_categories, industries, registered_date]).reset_index()

# unique()는 리스트가 아니라 numpy array로 나오니, 다루기 편하게 list로 변환
grouped["item_categories"] = grouped["item_categories"].apply(
    lambda x: list(x) if isinstance(x, (list, tuple)) or hasattr(x, "tolist") else []
)
grouped["industries"] = grouped["industries"].apply(
    lambda x: list(x) if isinstance(x, (list, tuple)) or hasattr(x, "tolist") else []
)

print(f"회사 기준 중복 제거: {len(df)}행 → {len(grouped)}행")
grouped.head()

## 5. 못 채우는 필드는 빈 값으로 명시 (다음 단계 클렌징 대상 표시)

- `supplier_type`: 나라장터 등록업체는 거의 법인이라 `"Company"`로 기본값만 채움 (드문 개인/공동사업자는 나중에 검증시 수정)
- `email`, `phone`: 이 데이터셋엔 아예 없음 → `None`으로 남겨서, DB클렌징(Tavily) 대상임을 표시


In [ ]:
grouped["supplier_type"] = "Company"
grouped["email"] = None
grouped["phone"] = None
grouped["needs_enrichment"] = True   # 클렌징 대상 표시 플래그

print(f"공급업체유형 채워짐: {grouped['supplier_type'].notna().sum()}건 (기본값)")
print(f"메일 채워짐: {grouped['email'].notna().sum()}건")
print(f"연락처 채워짐: {grouped['phone'].notna().sum()}건")

## 6. RAG용 설명문(description) 생성

⚠️ 품목명으로 검색해서 후보군을 뽑는 용도이므로, **취급품목(item_categories)을
설명문 앞쪽·비중있게 배치**해서 임베딩 시 의미 매칭이 잘 되도록 함.


In [ ]:
def make_description(row):
    parts = []
    if row["item_categories"]:
        parts.append("취급품목: " + ", ".join(row["item_categories"]))
    if row["industries"]:
        parts.append("업종: " + ", ".join(row["industries"]))
    parts.append(f"회사명: {row['company_name']}")
    return " / ".join(parts)

grouped["description"] = grouped.apply(make_description, axis=1)
grouped[["company_name", "description"]].head(10)

## 7. 최종 확인 및 저장

In [ ]:
print(f"최종 회사 수: {len(grouped)}개")
print(f"컬럼: {grouped.columns.tolist()}")
grouped.head()

In [ ]:
grouped.to_csv("vendors_preprocessed.csv", index=False, encoding="utf-8-sig")
print("'vendors_preprocessed.csv' 저장 완료 — 다음 단계: Postgres(pgvector) 적재")